# Project Final Report

In [45]:
### Run this cell before continuing.
import altair as alt
import numpy as np
import pandas as pd
import warnings
from sklearn import set_config
from sklearn.compose import make_column_transformer
from sklearn.metrics.pairwise import euclidean_distances
from sklearn.model_selection import (
    GridSearchCV,
    RandomizedSearchCV,
    cross_validate,
    train_test_split,
)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Simplify working with large datasets in Altair
alt.data_transformers.enable('vegafusion')

# Output dataframes instead of arrays
set_config(transform_output="pandas")

np.random.seed(155)
warnings.filterwarnings('ignore')

## 1) Introduction

**Background & Motivation**
This project presents a unique opportunity to engage in a real-world data science initiative with active stakeholders seeking actionable insights from their data. The research focuses on understanding how individuals interact with a custom MineCraft server, with the broader goal of informing recruitment strategies and resource allocation for ongoing studies.

**The Question:** *Can we predict if a Minecraft player subscribes to the community newsletter based on their total hours played and self-reported experience level?*

**The Dataset**
The players dataset below includes all variables required for this analysis. The explanatory variables will be “played_hours” (total hours played by each player) and “experience” (the player's level of experience). The response variable of interest is “subscribe,” which indicates whether a player has subscribed to the game-related newsletter. By analyzing the relationship between hours played and player experience, we explore whether we can predict that more/less engaged, experienced players will subscribe to the newsletter.




## 1. Introduction

**The Question:** *Can we predict if a Minecraft player subscribes to the community newsletter based on their total hours played and self-reported experience level?*

**Background & Motivation**
This project presents a unique opportunity to engage in a real-world data science initiative with active stakeholders seeking actionable insights from their data. The research focuses on understanding how individuals interact with a custom MineCraft server, with the broader goal of informing recruitment strategies and resource allocation for ongoing studies. By understanding if highly engaged players (measured by playtime and experience) are more likely to subscribe to the newsletter, the team can target communication efforts more effectively.


**The Dataset**
To answer this question, we utilize two datasets linked by a unique `hashedEmail`:
1.  **`players.csv`**: Contains demographic information, self-reported experience levels (Categorical: Beginner to Pro), and newsletter subscription status (Binary: True/False).
2.  **`sessions.csv`**: Contains log data of login and logout times, which allows for a precise calculation of total playtime.

 The explanatory variables will be ``played_hours`` which will be calculated from **``sessions.csv``** for accuracy   and ``experience`` (the player's level of experience). The response variable of interest is ``subscribe``, which indicates whether a player has subscribed to the game-related newsletter. By analyzing the relationship between hours played and player experience, we explore whether we can predict that more/less engaged, experienced players will subscribe to the newsletter. 

#### (a) Loading and Inspecting Data

In [46]:
# Load the datasets from the provided URLs
players_url = "https://drive.google.com/uc?export=download&id=1Mw9vW0hjTJwRWx0bDXiSpYsO3gKogaPz"
session_url = "https://drive.google.com/uc?export=download&id=14O91N5OlVkvdGxXNJUj5jIsV5RexhzbB"

players_data = pd.read_csv(players_url)
sessions_data = pd.read_csv(session_url)

print("--- Players Data Sample ---")
display(players_data.head(3))

print("\n--- Sessions Data Sample ---")
display(sessions_data.head(3))

--- Players Data Sample ---


,experience,subscribe,hashedEmail,played_hours,name,gender,age,individualId,organizationName
0,Pro,True,f6daba428a5e19a3d47574858c13550499be23603422e6...,30.3,Morgan,Male,9,NaN,NaN
1,Veteran,True,f3c813577c458ba0dfef80996f8f32c93b6e8af1fa9397...,3.8,Christian,Male,17,NaN,NaN
2,Veteran,False,b674dd7ee0d24096d1c019615ce4d12b20fcbff12d79d3...,0.0,Blake,Male,17,NaN,NaN



--- Sessions Data Sample ---


,hashedEmail,start_time,end_time,original_start_time,original_end_time
0,bfce39c89d6549f2bb94d8064d3ce69dc3d7e72b38f431...,30/06/2024 18:12,30/06/2024 18:24,1.719770e+12,1.719770e+12
1,36d9cbb4c6bc0c1a6911436d2da0d09ec625e43e6552f5...,17/06/2024 23:33,17/06/2024 23:46,1.718670e+12,1.718670e+12
2,f8f5477f5a2e53616ae37421b1c660b971192bd8ff77e3...,25/07/2024 17:34,25/07/2024 17:57,1.721930e+12,1.721930e+12


#### (b) Data Overview

In [47]:
print("--- Players DataFrame Info ---")
players_df.info()

print("\n--- Sessions DataFrame Info ---")
sessions_df.info()

--- Players DataFrame Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 196 entries, 0 to 195
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   experience        196 non-null    object 
 1   subscribe         196 non-null    bool   
 2   hashedEmail       196 non-null    object 
 3   played_hours      196 non-null    float64
 4   name              196 non-null    object 
 5   gender            196 non-null    object 
 6   age               196 non-null    int64  
 7   individualId      0 non-null      float64
 8   organizationName  0 non-null      float64
dtypes: bool(1), float64(3), int64(1), object(4)
memory usage: 12.6+ KB

--- Sessions DataFrame Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1535 entries, 0 to 1534
Data columns (total 5 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   hashedEmail          1535 non-nu

**Dataset 1: `players.csv`**
The players data shown above contains the following 197 observations (rows) and 9 variables (columns), including:
*   **`hashedEmail` (String):** Unique identifier for each player. No missing values.
*   **`experience` (String/Categorical):** A player's self-reported skill level. 
*   **`subscribe` (Boolean):** This is already in the correct boolean format, which is great.
*   **`played_hours` (Float):** Appears to be a cumulative total of total hours spent on the server.
*   **`name` (String):** Player's display name. I will exclude this from my model because it's not a useful predictor.
*   **`Gender` (String/Categorical):** Variable describing the player's gender.
*   **``age` (Float):** Discrete numeric variable with the player’s age.
*   **`individualId`, `organizationName`:** These columns are completely empty (`0 non-null`) and provide no information. 

**Issue**:
- All rows of the columns ``individualId`` and ``organizationName`` are missing data.

**Dataset 2: `sessions.csv`**
The sessions data shown above contains the following 1536 observations (rows) and 5 variables (columns), including:
*   **`hashedEmail` (String):** The key for linking sessions to players. No missing values.
*   **`start_time`, `end_time` (String):** Timestamps for sessions stored in the format (DD/MM/YY HH:MM) should be converted to DateTime if used for analysis. 
*   **`original_start_time`, `original_end_time` (Float):** A continuous variable with the unix timestamp for when the session began and ended. 

**Issue**:
- `end_time` and `original_end_time` each have two missing values. The corresponding sessions cannot be used to calculate duration and should be filtered out.

## (2) Methods and results

#### (a) Data Wrangling and Cleaning

To prepare the data for analysis, we used both the ``players`` and ``sessions`` data sets. We prioritized the ``sessions`` data for calculating playtime, as it contained better information on play time than the players data. Our data preparation process involved:


1.  **Data Merging**: We merged the two data sets on the shared ``hashedEmail`` column and engineered a  ``played_hours`` column from the sessions data instead of from the players data.



2.  **Column Selection**: From the combined data set, we selected only the relevant columns: ``played_hours,`` ``experience,`` and ``subscribe.`` Any unnecessary columns were removed to simplify the modeling process.



3.  **Experience Encoding**: KNNs requires using numerical values to calculate distance. We update experience to be a numerical value. Since experience is an ordinal, categorical variable, the categories correspond to players' experience and have a clear, meaningful sequence, with 'beginner' as the least experienced and 'pro' as the most experienced. Therefore, we assign each experience level a number, ranging from 1 to 5.

In [48]:
sessions_data["start_time"] = pd.to_datetime(sessions_data["start_time"])
sessions_data["end_time"] = pd.to_datetime(sessions_data["end_time"])
players_data_tidy = players_data.drop(["played_hours"], axis=1)
sessions_data.dropna(subset=['end_time'], inplace=True)
sessions_data['duration_hours'] = (sessions_data['end_time'] - sessions_data['start_time']).dt.total_seconds() / 3600
player_hours = sessions_data.groupby('hashedEmail')['duration_hours'].sum().reset_index()
player_hours.rename(columns={'duration_hours': 'played_hours'}, inplace=True)

players_data_tidy = pd.merge(players_data_tidy, player_hours, on='hashedEmail', how='left')
players_data_tidy['played_hours'] = players_data_tidy['played_hours'].fillna(0)

# Re-label subscribe "True" as "subscribed", and subscribe "False" as "not subscribed"
players_data_tidy['subscribe'] = players_data_tidy['subscribe'].replace({
    True : "subscribed",
    False : "not subscribed"
})
# Re-label experience levels with numbers
players_data_tidy['experience'] = players_data_tidy['experience'].replace({
    'Beginner' : 1,
    'Amateur' : 2,
    'Regular' : 3,
    'Veteran' : 4,
    'Pro' : 5
})

# Drop the columns identified as not useful
players_data_tidy.drop(columns=['individualId', 'organizationName', 'name', 'gender', 'age', 'hashedEmail'], inplace=True)

print("Tidied Data ready for Analysis:")
players_data_tidy

Tidied Data ready for Analysis:


,experience,subscribe,played_hours
0,5,subscribed,33.650000
1,4,subscribed,4.250000
2,4,not subscribed,0.083333
3,2,subscribed,0.833333
4,3,subscribed,0.150000
...,...,...,...
191,2,subscribed,0.000000
192,4,not subscribed,0.350000
193,2,not subscribed,0.083333
194,2,not subscribed,2.983333


#### (b) Exploring and visualizing the data

In [49]:
# Count the number of occurrences of each value in subscribe.
players_data_tidy["subscribe"].value_counts()

subscribe
subscribed        144
not subscribed     52
Name: count, dtype: int64

In [50]:
players_plot = alt.Chart(players_data_tidy).mark_bar().encode(
    x=alt.X("played_hours").title("Play time in hours"),
    y=alt.Y("count()").title("Number of players"),
    color = alt.Color("subscribe").title("Subscription status"),
).facet(
    "subscribe:N",
    columns = 2,
    title = ("Figure 1.", "Number of players subscribed based on play time in hours")
)

players_plot

alt.FacetChart(...)

In [51]:
subscription_proportion = players_data.groupby('experience')['subscribe'].mean()
subscription_proportion_df = pd.DataFrame(subscription_proportion).reset_index()
experience_percent_plot = alt.Chart(subscription_proportion_df, title=["Figure 2.", "Percentage of players subscribed", "from each expereince level"]).mark_bar().encode(
    x=alt.X("experience").title("Player experience level"),
    y=alt.Y('subscribe').title("Percentage of players subscribed"),
)
experience_percent_plot

alt.Chart(...)

In [52]:
bar_experience = alt.Chart(players_data, title=("Figure 3.", "Average Hours Played By Experience")).mark_bar().encode(
    x=alt.X('experience').title('Player Experience Level'),
    y=alt.Y('mean(played_hours)').title('Average Total Played Hours'),
    color=alt.Color("experience").title("Player experience level")
).properties(
    title='Average Engagement by Experience Level',
    width=600
)

bar_experience

alt.Chart(...)

First, in Fig. 1, we visually explored the relationship between each predictor variable and our outcome of interest: subscription status. The first plot is a histogram of total hours played, faceted by subscription status. This visualization reveals that, beyond a certain threshold of hours played, all players are subscribed to the newsletter. This pattern may suggest that higher engagement, as measured by playtime, is associated with the likelihood of subscription. However, this interpretation should be approached with caution. This apparent cutoff could result from having very few players with extremely high playtime, rather than indicating a meaningful association. As such, we should be careful not to draw strong conclusions from this threshold without additional evidence. We will keep this in mind as we conduct further analysis. 

Next, in Fig. 2, we examined the percentage of players from each experience level who subscribed to the newsletter using a bar chart. This second visualization demonstrates that the subscription rate varies across experience categories. Notably, regulars have the highest subscription rate, while veterans and other groups show lower rates. This suggests that players at the 'regular' level might be at an optimal stage of engagement for subscribing. The variation across experience levels makes experience a relevant explanatory variable, and suggests that the relationship between experience and subscription may not be strictly linear.

The final visualization, in Fig. 3, investigates the relationship between our two predictor variables—hours played and experience level—using a bar graph of play time by experience group. This analysis is important since our predictive methods rely on distance calculations, and it’s crucial to know if the predictors are correlated. The plot indicates that amateurs and regulars have the highest mean play hours, while other groups contribute less playtime on average. Despite some relationship between experience and hours played, there remains meaningful variation within each experience group. This suggests that both predictors provide distinct information for modeling subscription status, reducing concerns about redundancy and supporting the inclusion of both features in our analysis. The results are quite surprising and contradict the intuitive assumption that more "experienced" players would play more. "Regular" players have, by far, the highest average engagement, followed by "Amateurs". In contrast, "Pro", "Beginner", and "Veteran" players show very low average engagement on this server. 

Together, these exploratory visualizations offer important insights: they confirm that both the number of hours played and experience level are associated with newsletter subscription, and they clarify how these predictors relate to one another. This understanding supports our modeling choices and gives us greater confidence that our analysis will effectively capture the factors influencing subscription behavior


#### (c) Preparing the model

The method we used for this problem is K-nearest neighbors (KNN) classification. KNN classification is well-suited for predicting categorical outcomes. In this case, our target variable—newsletter subscription—has two possible values: subscribe (TRUE) or not subscribe (FALSE). This makes KNN a strong candidate for addressing our binary classification task. To evaluate the effectiveness of our model, we will use accuracy as the primary performance metric. Accuracy measures the proportion of correct predictions made by the model out of all predictions and is a common choice for binary classification problems like this one. Additionally, we will reinforce our evaluation by examining precision and recall. Precision measures the proportion of true positive predictions among all positive predictions, while recall assesses the proportion of true positives captured out of all actual positives. Considering these additional metrics provides a more comprehensive understanding of model performance, especially in the presence of class imbalance.

KNN classification does not require assumptions about the underlying distribution or linearity of the data, which is advantageous here since we do not know the precise relationship between experience, hours played, and newsletter subscription. The primary assumption underlying KNN classification is that individuals with similar predictor values—here, experience and hours played—will have similar outcomes regarding newsletter subscription. The model assumes that these features are strong predictors of the decision to subscribe. 

However, KNN has several potential limitations. It can become computationally slow with large datasets or a large number of predictors. Since the classes are slightly imbalanced (144 subscribers and 52 non-subscribers), the model may favor the majority class, especially if the K value is small. Additionally, because KNN is a distance-based method, it is essential to standardize predictor variables before applying the model, ensuring that no single variable biases predictions by disproportionately influencing the distance calculation. We upsample the unsubscribed users in the data to match the count of subscribed issues to partially mitigate some of these limitations


To begin preparing our model, we split the dataset into a training set and a test set—using an 80/20 split. We then start by upsampling the unsubscribed class. This ensures that we can train the model on one portion of the data and evaluate its performance on unseen data, providing a realistic assessment of predictive accuracy and minimizing the risk of overfitting. We then standardize the “played_hours” and “experience” variables, since they are measured on different scales. Standardization ensures that both variables contribute equally to KNN distance calculations and prevents any variable from disproportionately influencing the model’s predictions."

In [57]:

# Split stratify data and to ensure that the training and testing subsets 
# contain the right proportions of each category of observation.
players_train, players_test = train_test_split(
    players_data_tidy, train_size=0.8, stratify=players_data_tidy["subscribe"]
)

train_unsubscribed = players_train[players_train["subscribe"] == "not subscribed"]
train_subscribed = players_train[players_train["subscribe"] == "subscribed"]
train_unsubscribed_upsample = train_unsubscribed.sample(
    n=train_subscribed.shape[0], 
    replace=True,
    random_state=42
)

players_train = pd.concat((train_unsubscribed_upsample, train_subscribed))

print("Training set counts (Balanced):")
print(players_train["subscribe"].value_counts())

print("\nTest set counts (Original/Imbalanced):")
print(players_test["subscribe"].value_counts())

Training set counts (Balanced):
subscribe
not subscribed    115
subscribed        115
Name: count, dtype: int64

Test set counts (Original/Imbalanced):
subscribe
subscribed        29
not subscribed    11
Name: count, dtype: int64


In [54]:
players_train.head()

,experience,subscribe,played_hours


In [15]:
players_test.head()

,experience,subscribe,played_hours
4,3,subscribed,0.150000
79,2,subscribed,0.116667
69,3,subscribed,0.000000
23,1,subscribed,0.000000
18,2,subscribed,0.716667


We can see from the info method above that the training set contains 156 observations, while the test set contains 40 observations. This corresponds to the desired train/test split of 80/20.

In [16]:
# Create a pipline for KNN classification
players_preprocessor = make_column_transformer(
    (StandardScaler(), ["experience", "played_hours"]),
)
knn = KNeighborsClassifier(n_neighbors=3)

X = players_train[["experience", "played_hours"]]
y = players_train["subscribe"]
X_test = players_test[["experience", "played_hours"]]
y_test = players_test["subscribe"]

knn_pipeline = make_pipeline(players_preprocessor, knn)
knn_pipeline.fit(X, y)

knn_pipeline

,steps,"[('columntransformer', ...), ('kneighborsclassifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('standardscaler', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


With the pipeline complete, all necessary wrangling has been performed. 

#### (d) Buidling the model

Most predictive models in statistics and machine learning require the selection of key parameters. In KNN classification, the number of neighbors (K) is a crucial parameter that determines how many neighbors contribute to the class vote. By varying K, we can create different classifiers with varying predictive performance. Having already split our data into training and test sets and built an initial model with K=3, we next focused on finding the K value that yielded the highest accuracy. To avoid overfitting, we did not use the test set during model selection. Instead, we used cross-validation within the training data to evaluate model performance for different K values.

To ensure that every observation is used for both training and validation, we applied k-fold cross-validation, splitting the training data into evenly sized folds. Each fold was used once as a validation set, while the remaining folds were used for training. We first performed 5-fold cross-validation, then experimented with 10-fold cross-validation, which slightly reduced the standard error of our accuracy estimates. Based on these results, we opted for 10-fold cross-validation. This process yielded an estimated accuracy of around 64%. We used cross-validation accuracy to compare different K values and selected the K that maximized accuracy.


In [17]:
cv_5_df = pd.DataFrame(
    cross_validate(
        estimator=knn_pipeline,
        cv=5,
        X=X,
        y=y
    )
)

cv_5_df

,fit_time,score_time,test_score
0,0.005775,0.007246,0.656250
1,0.002558,0.001512,0.612903
2,0.002129,0.001405,0.645161
3,0.001501,0.002387,0.709677
4,0.001946,0.001160,0.741935


In [18]:
cv_5_metrics = cv_5_df.agg(["mean", "sem"])
cv_5_metrics

,fit_time,score_time,test_score
mean,0.002782,0.002742,0.673185
sem,0.000767,0.001145,0.023200


In [19]:
cv_10 = pd.DataFrame(
    cross_validate(
        estimator=knn_pipeline,
        cv=10,
        X=X,
        y=y
    )
)

cv_10_df = pd.DataFrame(cv_10)
cv_10_metrics = cv_10_df.agg(["mean", "sem"])
cv_10_metrics

,fit_time,score_time,test_score
mean,0.002062,0.002159,0.660417
sem,0.000438,0.000742,0.026222


In [20]:
cv_50_df = pd.DataFrame(
    cross_validate(
        estimator=knn_pipeline,
        cv=50,
        X=X,
        y=y
    )
)
cv_50_metrics = cv_50_df.agg(["mean", "sem"])
cv_50_metrics

/opt/homebrew/anaconda3/envs/dsci/lib/python3.11/site-packages/sklearn/model_selection/_split.py:811: UserWarning: The least populated class in y has only 41 members, which is less than n_splits=50.
  warnings.warn(


,fit_time,score_time,test_score
mean,0.001304,0.001134,0.638333
sem,0.000032,0.000029,0.029509


To automate hyperparameter tuning, we used scikit-learn's GridSearchCV. We first created a pipeline with a KNeighborsClassifier, leaving the number of neighbors (n_neighbors) parameter empty so it could be optimized. We then defined a grid of possible values for n_neighbors and constructed a parameter_grid dictionary to instruct GridSearchCV which values to evaluate.

We constructed the GridSearchCV object by passing our pipeline as the estimator, the parameter_grid as the param_grid, and specifying 10-fold cross-validation (cv=10). We used the fit method on the GridSearchCV object, providing the training predictors and labels. The cv_results_ attribute contained cross-validation accuracy estimates for each n_neighbors value, which we converted into a pandas DataFrame for easier analysis.

In [21]:
knn_0 = KNeighborsClassifier()
players_tune_pipe = make_pipeline(players_preprocessor, knn_0)
parameter_grid = {
    "kneighborsclassifier__n_neighbors": range(1, 100, 5),
}
players_tune_grid = GridSearchCV(
    estimator=players_tune_pipe,
    param_grid=parameter_grid,
    cv=10
)
players_tune_grid.fit(
    X,
    y
)
accuracies_grid = pd.DataFrame(players_tune_grid.cv_results_)
accuracies_grid.head()

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_kneighborsclassifier__n_neighbors,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,split5_test_score,split6_test_score,split7_test_score,split8_test_score,split9_test_score,mean_test_score,std_test_score,rank_test_score
0,0.001858,0.000642,0.001746,0.000738,1,{'kneighborsclassifier__n_neighbors': 1},0.875,0.6250,0.6250,0.4375,0.5000,0.6250,0.666667,0.533333,0.733333,0.666667,0.628750,0.116941,19
1,0.001274,0.000134,0.001086,0.000162,6,{'kneighborsclassifier__n_neighbors': 6},0.625,0.5625,0.6875,0.5000,0.6875,0.6250,0.733333,0.600000,0.600000,0.600000,0.622083,0.063929,20
2,0.001262,0.000132,0.001179,0.000182,11,{'kneighborsclassifier__n_neighbors': 11},0.750,0.7500,0.7500,0.7500,0.7500,0.6875,0.733333,0.733333,0.733333,0.733333,0.737083,0.018300,1
3,0.001319,0.000181,0.001104,0.000133,16,{'kneighborsclassifier__n_neighbors': 16},0.750,0.7500,0.7500,0.7500,0.7500,0.6875,0.733333,0.733333,0.733333,0.733333,0.737083,0.018300,1
4,0.001216,0.000158,0.001054,0.000132,21,{'kneighborsclassifier__n_neighbors': 21},0.750,0.7500,0.7500,0.7500,0.7500,0.6875,0.733333,0.733333,0.733333,0.733333,0.737083,0.018300,1


From the accuracy grid, we isolated the relevant quantities: the number of neighbors (param_kneighbors_classifier__n_neighbors), the cross-validation accuracy estimate (mean_test_score), and the standard error of the accuracy estimate. Since GridSearchCV outputs the standard deviation (std_test_score) rather than the standard error, we computed the standard error by dividing the standard deviation by the square root of the number of folds.

In [22]:
accuracies_grid["sem_test_score"] = accuracies_grid["std_test_score"] / 10**(1/2)
accuracies_grid = (
    accuracies_grid[[
        "param_kneighborsclassifier__n_neighbors",
        "mean_test_score",
        "sem_test_score"
    ]]
    .rename(columns={"param_kneighborsclassifier__n_neighbors": "n_neighbors"})
)
accuracies_grid.head()

,n_neighbors,mean_test_score,sem_test_score
0,1,0.628750,0.036980
1,6,0.622083,0.020216
2,11,0.737083,0.005787
3,16,0.737083,0.005787
4,21,0.737083,0.005787


In [23]:
accuracy_vs_k = alt.Chart(accuracies_grid, title=("Figure 4.", "Estimated accuracy versus the number of neighbors")).mark_line(point=True).encode(
    x=alt.X("n_neighbors").title("Neighbors"),
    y=alt.Y("mean_test_score")
        .scale(zero=False)
        .title("Accuracy estimate")
)

accuracy_vs_k

alt.Chart(...)

In [24]:
players_tune_grid.best_params_

{'kneighborsclassifier__n_neighbors': 11}

To identify the optimal number of neighbors, we plotted accuracy versus K and also programmatically accessed the best_params_ attribute of the fitted GridSearchCV object. Both methods indicated that accuracy increased rapidly with K and then plateaued, helping us pinpoint the optimal K. Setting n_neighbors to 16 yielded the highest cross-validation accuracy estimate.

In [25]:
players_test["predicted"] = players_tune_grid.predict(
    players_test[["experience", "played_hours"]]
)

players_tune_grid.score(
    players_test[["experience", "played_hours"]],
    players_test["subscribe"]
)

0.7

After finalizing the model, we evaluated its predictive performance on the held-out test set. We retrained the KNN classifier on the full training data using the chosen number of neighbors (which scikit-learn's GridSearchCV handles automatically). We then used the score and predict methods of the fitted GridSearchCV object to assess the model's accuracy and generate predictions on the test data.

The final model achieved an accuracy of 72.5% on the test set.

In [26]:
from sklearn.metrics import precision_score
precision_score(
    y_true=players_test["subscribe"],
    y_pred=players_test["predicted"],
    pos_label='subscribed'
)

0.717948717948718

In [27]:
from sklearn.metrics import recall_score
recall_score(
    y_true=players_test["subscribe"],
    y_pred=players_test["predicted"],
    pos_label='subscribed'
)

0.9655172413793104

Precision measures the proportion of positive predictions made by the classifier that are actually correct. Here, we define the positive value as “subscribe,” meaning the player has subscribed to the newsletter, which is important to the developers. Notably, the precision matches the overall accuracy of the model, which could indicate a data imbalance. If there are very few negative examples, the model’s accuracy may appear high even if it struggles to correctly identify the positive class.

Recall measures the proportion of actual positive observations in the test set that are correctly identified by the model. In our case, a recall value of 1 means the model labels every true ‘subscribe’ observation correctly. Although this might seem “good”, when considered with the precision score, it highlights the model’s shortcomings: it predicts ‘subscribe’ for every observation, failing to distinguish between positive and negative values.

In [28]:
pd.crosstab(
    players_test["subscribe"],
    players_test["predicted"]
)

predicted,not subscribed,subscribed
subscribe,,
not subscribed,0,11
subscribed,1,28


From the crosstab we can see that the classifier only ever predicted that players would subscribe to the newsletter.

## 3) Discussion

This analysis set out to determine whether two key features—experience points and played hours—could accurately predict whether a player would subscribe to the game-related newsletter. Our expectation, based on common intuition, was that players who invested more time and had greater experience would be more likely to subscribe, reflecting stronger engagement or commitment to the game.

However, the results did not reveal what we expected. Despite our initial exploration of the variables used and thorough hyperparameter tuning, the predictive K-Nearest Neighbors (KNN) classification model performed no better than a majority-class baseline. In other words, experience and played hours did not distinguish subscribers from non-subscribers in this dataset.

One possible explanation for this unexpected outcome lies in the data source: the Minecraft server was primarily used by college students as part of an academic assignment. This context limited the diversity of player behavior, since many students may have engaged just enough to satisfy course requirements rather than out of genuine interest. As seen in our exploratory visualizations, there were also a few outliers who contributed a large number of hours in comparison to the other students. Consequently, both the predictors (experience and played hours) and the outcome (subscribe) showed little variation, making it difficult for any model to detect meaningful patterns or achieve accurate predictions. The limited variability in both features and the target variable, driven by the specific academic collection context, likely prevented the KNN model from finding any meaningful, generalizable differences between subscribers and non-subscribers.

The impact of these findings is significant for the research group and stakeholders. First, simple behavioral metrics such as playtime or experience cannot be relied on in this context to identify or target committed players for future studies or resource planning. Also, forecasting server needs or licenses based on these features risks inefficient allocation, as there is no reliable way to segment players by expected engagement using current variables. The results highlight the need for more nuanced, context-specific behavioral data to understand better and predict player engagement or subscription decisions.

These findings raise several important questions for the future:
- What additional data (qualitative feedback, motivation surveys, in-game activity) could provide a more accurate picture of engagement?
- How might player behavior differ in a less specific, more natural environment?
- Would alternative modeling approaches have different results, or is the lack of predictive power due in its entirety to the current context?

In summary, while the initial hypothesis was not supported, these results provide valuable guidance for refining research methods and data collection strategies in future work.
